# Комп'ютерна графіка — тренажер «Три проекції»

NO-CAD: задаємо просторовий об'єкт координатами, а система автоматично будує його ортогональні проекції.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox

POINT_COLORS = {'A': 'tab:red', 'B': 'tab:blue', 'C': 'tab:green'}

## 1. Просторова модель

Три точки визначають площину: A(x, y, z), B(x, y, z), C(x, y, z).

In [ ]:
def make_points(Ax, Ay, Az, Bx, By, Bz, Cx, Cy, Cz):
    return np.array([
        [Ax, Ay, Az],
        [Bx, By, Bz],
        [Cx, Cy, Cz]
    ], dtype=float)

## 2. Визначення положення площини

In [ ]:
def plane_type(P):
    A, B, C = P
    u = B - A
    v = C - A
    normal = np.cross(u, v)

    if np.linalg.norm(normal) < 1e-9:
        return 'Площина не визначена: точки A, B, C колінеарні.'

    if np.allclose(P[:, 2], P[0, 2]):
        return 'Горизонтальна площина: z = const'

    if np.allclose(P[:, 1], P[0, 1]):
        return 'Фронтальна площина: y = const'

    if np.allclose(P[:, 0], P[0, 0]):
        return 'Профільна площина: x = const'

    return 'Площина загального положення'

## 3. Масштаб координатних полів

In [ ]:
def get_limits(P):
    low = P.min(axis=0)
    high = P.max(axis=0)
    span = np.maximum(high - low, 2)
    return low - 0.35 * span, high + 0.35 * span

## 4. Побудова ортогональної проекції

Фронтальна: XZ. Горизонтальна: XY. Профільна: YZ.

In [ ]:
def draw_projection(ax, P, dims, title, show_lines=True):
    i, j = dims
    names = ['X', 'Y', 'Z']
    low, high = get_limits(P)
    Q = P[:, [i, j]]

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(names[i])
    ax.set_ylabel(names[j])
    ax.set_xlim(low[i], high[i])
    ax.set_ylim(low[j], high[j])
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.25)

    ax.fill(Q[:, 0], Q[:, 1], alpha=0.10)
    ax.plot(Q[[0, 1, 2, 0], 0], Q[[0, 1, 2, 0], 1], color='black')

    for k, name in enumerate(['A', 'B', 'C']):
        ax.scatter(Q[k, 0], Q[k, 1], s=70, color=POINT_COLORS[name], zorder=3)
        ax.annotate(name, (Q[k, 0], Q[k, 1]), xytext=(7, 7), textcoords='offset points')

        if show_lines:
            ax.axvline(Q[k, 0], color=POINT_COLORS[name], linestyle='--', alpha=0.18)
            ax.axhline(Q[k, 1], color=POINT_COLORS[name], linestyle='--', alpha=0.18)

## 5. Допоміжний 3D-вигляд

In [ ]:
def draw_3d(ax, P):
    ax.set_title('Допоміжний 3D-вигляд', fontweight='bold')

    ax.plot(
        P[[0, 1, 2, 0], 0],
        P[[0, 1, 2, 0], 1],
        P[[0, 1, 2, 0], 2],
        color='black'
    )

    ax.plot_trisurf(P[:, 0], P[:, 1], P[:, 2], alpha=0.12)

    for k, name in enumerate(['A', 'B', 'C']):
        ax.scatter(*P[k], s=70, color=POINT_COLORS[name])
        ax.text(P[k, 0], P[k, 1], P[k, 2], '  ' + name)

    low, high = get_limits(P)
    ax.set_xlim(low[0], high[0])
    ax.set_ylim(low[1], high[1])
    ax.set_zlim(low[2], high[2])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.view_init(elev=24, azim=-58)

## 6. Основна функція тренажера

In [ ]:
def show_trainer(
    Ax=2, Ay=2, Az=5,
    Bx=8, By=2, Bz=5,
    Cx=5, Cy=7, Cz=5,
    show_lines=True
):
    P = make_points(Ax, Ay, Az, Bx, By, Bz, Cx, Cy, Cz)

    fig = plt.figure(figsize=(15, 9))
    ax_front = fig.add_subplot(221)
    ax_profile = fig.add_subplot(222)
    ax_horizontal = fig.add_subplot(223)
    ax_3d = fig.add_subplot(224, projection='3d')

    draw_projection(ax_front, P, (0, 2), 'Фронтальна проекція XZ', show_lines)
    draw_projection(ax_profile, P, (1, 2), 'Профільна проекція YZ', show_lines)
    draw_projection(ax_horizontal, P, (0, 1), 'Горизонтальна проекція XY', show_lines)
    draw_3d(ax_3d, P)

    fig.suptitle(plane_type(P), fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()


interact(
    show_trainer,
    Ax=FloatSlider(min=0, max=10, step=1, value=2, description='A: X'),
    Ay=FloatSlider(min=0, max=10, step=1, value=2, description='A: Y'),
    Az=FloatSlider(min=0, max=10, step=1, value=5, description='A: Z'),
    Bx=FloatSlider(min=0, max=10, step=1, value=8, description='B: X'),
    By=FloatSlider(min=0, max=10, step=1, value=2, description='B: Y'),
    Bz=FloatSlider(min=0, max=10, step=1, value=5, description='B: Z'),
    Cx=FloatSlider(min=0, max=10, step=1, value=5, description='C: X'),
    Cy=FloatSlider(min=0, max=10, step=1, value=7, description='C: Y'),
    Cz=FloatSlider(min=0, max=10, step=1, value=5, description='C: Z'),
    show_lines=Checkbox(value=True, description='Проекційні лінії')
);

## 7. Готові приклади

In [ ]:
# Горизонтальна площина: z = const
show_trainer(2, 2, 5, 8, 2, 5, 5, 7, 5)

In [ ]:
# Фронтальна площина: y = const
show_trainer(2, 4, 2, 8, 4, 2, 5, 4, 7)

In [ ]:
# Профільна площина: x = const
show_trainer(4, 2, 2, 4, 8, 2, 4, 5, 7)

In [ ]:
# Площина загального положення
show_trainer(2, 2, 2, 8, 3, 6, 5, 8, 4)

## 8. Навчальне завдання

1. Встановіть горизонтальну площину.
2. Змініть координату Z однієї точки.
3. Простежте, як змінилися три проекції.
4. Отримайте фронтальну та профільну площини.
5. Створіть площину загального положення.
6. Сформулюйте правило: яка координата є сталою для кожного спеціального положення площини?